# Are Topics Evidence? Coherence and Its Critics

**Computational Text Analysis Pack · Module 08 — Topic Modelling**

*Technique:* NPMI coherence; Chang's 'reading tea leaves'; Schmidt's 'Words Alone'

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/05-unsupervised-structure/08d_are_topics_evidence.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Notebook 08c read topics sceptically. This one asks the harder question the
survey puts at the centre of the topic-modelling literature: **can a topic model
be evidence in a scholarly argument at all?**

Three findings frame it:

- **Chang et al. (2009), "Reading tea leaves"** — automated coherence scores can
  *anti-correlate* with human judgements of interpretability.
- **Hoyle et al. (2021)** — automated topic-model evaluation is, in their word,
  broken.
- **Ben Schmidt, "Words Alone" (2012)** — judging a topic by its top words is
  dangerous; a model has to be read as a whole set and grounded in the words it
  was built from.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

sub = df[df["language"] == "English"].reset_index(drop=True)
vec = CountVectorizer(token_pattern=r"[\w\u0300-\u036f]+", min_df=5, max_df=0.4, max_features=3000)
X = vec.fit_transform(sub["text"])
vocab = np.array(vec.get_feature_names_out())

K = 10                      # <- change this
lda = LatentDirichletAllocation(n_components=K, random_state=0, max_iter=25)
lda.fit(X)
print(f"{X.shape[0]} documents, {X.shape[1]} vocabulary, {K} topics")

## Computing coherence rather than trusting it

NPMI coherence asks: do a topic's top words actually co-occur in documents? It
is the standard automated score, and computing it ourselves is the only way to
see what it does and does not capture.

In [ ]:
binary = (X > 0).astype(int)
doc_freq = np.asarray(binary.sum(axis=0)).ravel()
n_docs = binary.shape[0]

def npmi_coherence(topic_idx, top_n=10):
    top = np.argsort(lda.components_[topic_idx])[::-1][:top_n]
    scores = []
    for a in range(len(top)):
        for b in range(a + 1, len(top)):
            i, j = top[a], top[b]
            co = int(binary[:, i].multiply(binary[:, j]).sum())
            p_i, p_j = doc_freq[i] / n_docs, doc_freq[j] / n_docs
            p_ij = co / n_docs
            if p_ij == 0:
                scores.append(-1.0)
                continue
            scores.append(np.log(p_ij / (p_i * p_j)) / -np.log(p_ij))
    return float(np.mean(scores))

topics = []
for k in range(K):
    top = np.argsort(lda.components_[k])[::-1][:10]
    topics.append({"topic": k, "coherence": npmi_coherence(k),
                   "top_words": " ".join(vocab[top])})
tt = pd.DataFrame(topics).sort_values("coherence", ascending=False)
print(tt.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

## The Chang test, in miniature

Chang's method was the **word intrusion** task: take a topic's top words, swap
one for a word that does not belong, and see whether a human can spot the
intruder. If the topic is genuinely coherent, the intruder is obvious.

We can build the task here. Run the cell, try it yourself before looking at the
answer, and then compare your success rate against the coherence ranking.

In [ ]:
rng = np.random.default_rng(7)
print("WORD INTRUSION — one word in each row does not belong.\n")
truth = {}
for k in range(K):
    top = list(np.argsort(lda.components_[k])[::-1][:5])
    # An intruder: low probability in THIS topic, high in some other.
    other = rng.choice([j for j in range(K) if j != k])
    cand = [w for w in np.argsort(lda.components_[other])[::-1][:20]
            if w not in top]
    intruder = int(cand[0])
    words = [vocab[i] for i in top] + [vocab[intruder]]
    order = rng.permutation(len(words))
    shown = [words[i] for i in order]
    truth[k] = vocab[intruder]
    print(f"  topic {k}:  " + "   ".join(shown))
print("\n(answers in the next cell)")

In [ ]:
print(f"{'topic':<8}{'intruder':<16}{'coherence':>11}")
print("-" * 35)
for k in range(K):
    c = float(tt.loc[tt['topic'] == k, 'coherence'].iloc[0])
    print(f"{k:<8}{truth[k]:<16}{c:>11.3f}")
print()
print("Chang's finding was that the ordering by coherence and the ordering by")
print("human intrusion-detection do not reliably agree. If you found an intruder")
print("easy to spot in a LOW-coherence topic, or hard in a HIGH-coherence one,")
print("you have just reproduced the result on this corpus.")

## Schmidt's objection: read the whole set

Every table above shows ten words per topic. Schmidt's argument is that this is
the problem, not the presentation: the top words are a *summary* of a
distribution over thousands of words, and summaries are where interpretation
quietly happens.

The cell below shows how much of each topic those ten words actually account
for.

In [ ]:
print(f"{'topic':<8}{'mass in top 10':>16}{'words for 50% of mass':>24}")
print("-" * 48)
for k in range(K):
    dist = lda.components_[k] / lda.components_[k].sum()
    ordered = np.sort(dist)[::-1]
    top10 = ordered[:10].sum()
    half = int(np.searchsorted(np.cumsum(ordered), 0.5) + 1)
    print(f"{k:<8}{top10:>16.1%}{half:>24,}")
print()
print("The ten words you were shown carry a small fraction of the topic. The")
print("rest of the distribution is doing most of the work, unseen.")

### 🔧 Your turn 1

Raise K to 25 and re-run. Do the coherence scores rise or fall, and does the word-intrusion task get easier or harder?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Coherence usually *rises* with more topics, because each topic gets narrower and
its top words co-occur more tightly. Intrusion often gets harder, because narrow
topics start to overlap and an intruder from a neighbouring topic looks
plausible. That divergence — the automated score improving while human
interpretability degrades — is Chang's result and Hoyle's warning in one
experiment, and it is why "we chose k by maximising coherence" is not a
defensible methods sentence on its own.

</details>